# Linear algebra with `Mat`

`Mat` holds a matrix in a pool and knows the general linear algebra:
`Solve` (LU), `Det`, `Inverse`, `Qr`, `LstSq`, `Svd`, `EigSym`, each
held to numpy at 1e-13. A matrix is opaque, so the first cell is a
**library cell** -- a definition with its implementation, no state --
that prints one; every later cell imports it.

In [ ]:
DEFINITION MODULE Show ;

IMPORT Mat ;

PROCEDURE Matrix (RO name: STR ; m: PTR Mat.Matrix) RAISES ValueRange ;
  (* the matrix, a row a line, four decimals *)

PROCEDURE Vector (RO name: STR ; RO v: SLICE OF F64) RAISES ValueRange ;

END Show.

IMPLEMENTATION MODULE Show ;

IMPORT Fmt ;
IMPORT Io ;
IMPORT Mat ;

PROCEDURE Matrix (RO name: STR ; m: PTR Mat.Matrix) RAISES ValueRange =
VAR
  r, c : I64 ;
  line : STR ;
BEGIN
  Io.WriteLine (name + ':') ;
  FOR r := 0 TO Mat.Rows (m) - 1 DO
    line := '  ' ;
    FOR c := 0 TO Mat.Cols (m) - 1 DO
      line := line + Fmt.FixedPad (Mat.Get (m, r, c), 10, 4)
    END ;
    Io.WriteLine (line)
  END
END Matrix ;

PROCEDURE Vector (RO name: STR ; RO v: SLICE OF F64) RAISES ValueRange =
VAR
  i : I64 ;
  line : STR ;
BEGIN
  line := name + ':' ;
  FOR i := 0 TO LEN (v) - 1 DO
    line := line + Fmt.FixedPad (v[i], 10, 4)
  END ;
  Io.WriteLine (line)
EXCEPT
| IndexError :
    Io.ErrLine ('out of range')
END Vector ;

END Show.

A system of three equations: `Solve` answers `x` with `A x = b`, and
the determinant and the inverse are checked against each other.

In [ ]:
MODULE Solve ;

IMPORT Faults ;
IMPORT Fmt ;
IMPORT Io ;
IMPORT Mat ;
IMPORT Show ;

VAR
  pool : POOL ;
  a, b, x, inv, id : PTR Mat.Matrix IN pool ;

BEGIN
  a := Mat.New (pool, 3, 3) ;
  Mat.Set (a, 0, 0, 4.0) ; Mat.Set (a, 0, 1, -2.0) ; Mat.Set (a, 0, 2, 1.0) ;
  Mat.Set (a, 1, 0, -2.0) ; Mat.Set (a, 1, 1, 4.0) ; Mat.Set (a, 1, 2, -2.0) ;
  Mat.Set (a, 2, 0, 1.0) ; Mat.Set (a, 2, 1, -2.0) ; Mat.Set (a, 2, 2, 4.0) ;
  b := Mat.New (pool, 3, 1) ;
  Mat.Set (b, 0, 0, 11.0) ; Mat.Set (b, 1, 0, -16.0) ; Mat.Set (b, 2, 0, 17.0) ;
  Show.Matrix ('A', a) ;
  x := Mat.Solve (pool, a, b) ;
  Show.Matrix ('x, with A x = b', x) ;
  Io.WriteLine ('det A = ' + Fmt.Fixed (Mat.Det (a), 4)) ;
  inv := Mat.Inverse (pool, a) ;
  Show.Matrix ('inverse of A', inv) ;
  id := Mat.MulM (pool, a, inv) ;
  Show.Matrix ('A times its inverse', id)
EXCEPT
| Mat.Singular :
    Io.ErrLine ('A is singular')
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| ValueRange :
    Io.ErrLine ('a value out of range')
END Solve.

Factorisations: `Qr` (Householder, R's diagonal never negative),
`Svd` (one-sided Jacobi, values descending) and `EigSym` (Jacobi,
values ascending, each vector's largest component positive -- the
sign is said, so two implementations agree to the digit).

In [ ]:
MODULE Factor ;

IMPORT Faults ;
IMPORT Io ;
IMPORT Mat ;
IMPORT Show ;

VAR
  pool : POOL ;
  a, q, r, u, vt, v : PTR Mat.Matrix IN pool ;
  s, w : SLICE OF F64 ;

BEGIN
  a := Mat.New (pool, 3, 3) ;
  Mat.Set (a, 0, 0, 2.0) ; Mat.Set (a, 0, 1, 1.0) ; Mat.Set (a, 0, 2, 0.0) ;
  Mat.Set (a, 1, 0, 1.0) ; Mat.Set (a, 1, 1, 3.0) ; Mat.Set (a, 1, 2, 1.0) ;
  Mat.Set (a, 2, 0, 0.0) ; Mat.Set (a, 2, 1, 1.0) ; Mat.Set (a, 2, 2, 4.0) ;
  Show.Matrix ('A (symmetric)', a) ;
  Mat.Qr (pool, a, q, r) ;
  Show.Matrix ('Q', q) ;
  Show.Matrix ('R', r) ;
  s := NEW (pool, F64, 3) ;
  Mat.Svd (pool, a, u, s, vt) ;
  Show.Vector ('singular values', s) ;
  w := NEW (pool, F64, 3) ;
  Mat.EigSym (pool, a, w, v) ;
  Show.Vector ('eigenvalues', w) ;
  Show.Matrix ('eigenvectors, a column each', v)
EXCEPT
| Mat.NoConverge :
    Io.ErrLine ('no convergence')
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| ValueRange :
    Io.ErrLine ('a value out of range')
END Factor.

Least squares: a parabola through noisy points is `LstSq` on the
design matrix `[1, t, t^2]` -- QR underneath, a rank-deficient design
refused by name as `Singular`.

In [ ]:
MODULE Parabola ;

IMPORT Faults ;
IMPORT Fmt ;
IMPORT Io ;
IMPORT Mat ;
IMPORT Plot ;
IMPORT Show ;
IMPORT Stats ;

CONST N = 40 ;

VAR
  pool : POOL ;
  st : Stats.Stream ;
  a, b, c : PTR Mat.Matrix IN pool ;
  t, y, fit : SLICE OF F64 ;
  i : I64 ;
  tt : F64 ;

BEGIN
  st := Stats.Seed (3) ;
  t := NEW (pool, F64, N) ;
  y := NEW (pool, F64, N) ;
  a := Mat.New (pool, N, 3) ;
  b := Mat.New (pool, N, 1) ;
  FOR i := 0 TO N - 1 DO
    tt := F64 (i) * 0.25 - 5.0 ;
    t[i] := tt ;
    y[i] := 0.5 * tt * tt - 2.0 * tt + 1.0 + Stats.Normal (st) ;
    Mat.Set (a, i, 0, 1.0) ; Mat.Set (a, i, 1, tt) ; Mat.Set (a, i, 2, tt * tt) ;
    Mat.Set (b, i, 0, y[i])
  END ;
  c := Mat.LstSq (pool, a, b) ;
  Show.Matrix ('coefficients of 1, t, t^2 (true: 1, -2, 0.5)', c) ;
  fit := NEW (pool, F64, N) ;
  FOR i := 0 TO N - 1 DO
    fit[i] := Mat.Get (c, 0, 0) + Mat.Get (c, 1, 0) * t[i] + Mat.Get (c, 2, 0) * t[i] * t[i]
  END ;
  Plot.ClearFigure () ;
  Plot.SetDots (t, y) ;
  Plot.AddLine (t, fit, 1, 'least squares') ;
  Io.WriteFile ('parabola.svg', Plot.Render ('a parabola by least squares', 't', 'y'))
EXCEPT
| Mat.Singular :
    Io.ErrLine ('the design is rank-deficient')
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| ValueRange :
    Io.ErrLine ('a value out of range')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
END Parabola.